# 🔮 Digital Twin AI — EDA & Model Training

This notebook walks through:
1. Loading the NASA CMAPSS Turbofan Engine Degradation dataset
2. Exploratory Data Analysis (EDA)
3. Feature engineering & preprocessing
4. Training an XGBoost model for RUL prediction
5. Saving the model and scaler for the FastAPI backend

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import xgboost as xgb
import joblib
import os

plt.style.use('dark_background')
sns.set_palette('viridis')
print('Libraries loaded ✅')

## 1. Load Dataset

Download the CMAPSS dataset from [Kaggle](https://www.kaggle.com/datasets/behrad3d/nasa-cmaps) and place `train_FD001.txt`, `test_FD001.txt`, `RUL_FD001.txt` in `../data/raw/`.

In [ ]:
# Column names for CMAPSS
index_cols = ['unit_id', 'cycle']
setting_cols = ['op_setting_1', 'op_setting_2', 'op_setting_3']
sensor_cols = [f'sensor_{i}' for i in range(1, 22)]
all_cols = index_cols + setting_cols + sensor_cols

# Load training data
train_df = pd.read_csv('../data/raw/train_FD001.txt', sep=' ', header=None, names=all_cols,
                        index_col=False).dropna(axis=1)

print(f'Training set: {train_df.shape}')
print(f'Engines: {train_df.unit_id.nunique()}')
train_df.head()

## 2. Compute RUL (Target Variable)

In [ ]:
# RUL = max_cycle_per_engine - current_cycle
max_cycles = train_df.groupby('unit_id')['cycle'].max().reset_index()
max_cycles.columns = ['unit_id', 'max_cycle']
train_df = train_df.merge(max_cycles, on='unit_id')
train_df['RUL'] = train_df['max_cycle'] - train_df['cycle']

# Cap RUL at 125 (piece-wise linear)
RUL_CAP = 125
train_df['RUL'] = train_df['RUL'].clip(upper=RUL_CAP)

print(f'RUL range: {train_df.RUL.min()} - {train_df.RUL.max()}')
train_df[['unit_id', 'cycle', 'RUL']].head(10)

## 3. EDA — Sensor Distributions

In [ ]:
# Drop constant sensors
sensor_std = train_df[sensor_cols].std()
constant_sensors = sensor_std[sensor_std < 0.01].index.tolist()
useful_sensors = [s for s in sensor_cols if s not in constant_sensors]

print(f'Constant (dropped): {constant_sensors}')
print(f'Useful sensors: {len(useful_sensors)}')

fig, axes = plt.subplots(4, 4, figsize=(16, 12))
for i, sensor in enumerate(useful_sensors[:16]):
    ax = axes[i // 4, i % 4]
    for uid in [1, 50, 100]:
        engine = train_df[train_df.unit_id == uid]
        ax.plot(engine.cycle, engine[sensor], alpha=0.7, linewidth=0.8)
    ax.set_title(sensor, fontsize=9)
    ax.tick_params(labelsize=7)
plt.suptitle('Sensor Degradation Patterns', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 4. Feature Engineering & Scaling

In [ ]:
# Select features
feature_cols = useful_sensors
X = train_df[feature_cols].values
y = train_df['RUL'].values

# Scale features
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

print(f'Train: {X_train.shape}, Test: {X_test.shape}')

## 5. Train XGBoost Model

In [ ]:
model = xgb.XGBRegressor(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='reg:squarederror',
    random_state=42,
    n_jobs=-1,
)

model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=20)

# Evaluate
y_pred = model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f'\n📊 Results:')
print(f'  RMSE: {rmse:.2f}')
print(f'  MAE:  {mae:.2f}')
print(f'  R²:   {r2:.4f}')

## 6. Save Model & Scaler

In [ ]:
MODEL_DIR = '../models'
os.makedirs(MODEL_DIR, exist_ok=True)

joblib.dump(model, os.path.join(MODEL_DIR, 'turbo_model.pkl'))
joblib.dump(scaler, os.path.join(MODEL_DIR, 'scaler.joblib'))

print('✅ Model saved to models/turbo_model.pkl')
print('✅ Scaler saved to models/scaler.joblib')